# **Lexora Quiz Training Model**
### Lyanne Nanjala
###### This model aims to train the model to produce serious questions aimed at ensuring the application's user can get the full experience of the quiz feature. Using a dataset I created by compiling KCSE questions and questions from Strathmore's ExamBank, it is my hope that this model can use it to create similar structured questions designed to provide an educating experience to the user.
##### If it does not work, well, I tried my best.
##### Peace!


### ***Step 1: Import the necessary libraries***


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
import torch
import csv

In [ ]:
import os
import glob

def find_csv_file():
    """Search for the CSV file in common locations"""
    possible_paths = [
        'question_dataset.csv',
        'ai_services/question_dataset.csv',
        '../question_dataset.csv',
        './question_dataset.csv',
        'backend/ai_services/question_dataset.csv',
        'backend/question_dataset.csv',
        '../backend/ai_services/question_dataset.csv',
       
    ]
    
    search_patterns = [
        '**/question_dataset.csv',
        '**/*question_dataset*',
        '**/*.csv'
    ]
    
    print("Searching for CSV file...")
    
 
    for path in possible_paths:
        if os.path.exists(path):
            print(f"✓ Found file at: {path}")
            return path
    
   # Search using glob patterns
    for pattern in search_patterns:
        matches = glob.glob(pattern, recursive=True)
        for match in matches:
            if 'question' in match.lower():
                print(f"✓ Found potential file: {match}")
                return match
    
    
    print("\nAll CSV files found:")
    csv_files = glob.glob('**/*.csv', recursive=True)
    for csv_file in csv_files:
        print(f"  - {csv_file}")
    
    return None

# Find the file
file_path = find_csv_file()

if file_path:
    print(f"\nUsing file: {file_path}")
else:
    print("\n❌ CSV file not found. Please check:")
    print("1. The file exists and is named 'question_dataset.csv'")
    print("2. You're running the notebook from the correct directory")
    print("3. The file is in your project folder")

Searching for CSV file...
✓ Found file at: question_dataset.csv

Using file: question_dataset.csv


In [9]:
import os

print("Current working directory:", os.getcwd())
print("Files in current directory:")
for file in os.listdir('.'):
    print(f"  - {file}")

print("\nFiles in backend directory (if it exists):")
if os.path.exists('backend'):
    for file in os.listdir('backend'):
        print(f"  - backend/{file}")

Current working directory: c:\FinalProject\backend\ai_services
Files in current directory:
  - ai_companion.py
  - content_moderator.py
  - question_dataset.csv
  - quiz_generator.py
  - remove.ipynb
  - train_quiz_model.ipynb

Files in backend directory (if it exists):


### ***Step 2: Dataset is loaded***

In [ ]:
def load_dataset_safely(file_path):
    try:
       
        df = pd.read_csv(file_path, encoding='utf-8')
        print("✓ Dataset loaded successfully with standard method")
        return df
    except pd.errors.ParserError as e:
        print(f"Parser error: {e}")
        print("Trying alternative loading method...")
        
        # Alternative method
        data = []
        with open(file_path, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            
            try:
                headers = next(reader)
                print(f"Detected headers: {headers}")
                
                for i, row in enumerate(reader):
                    # Handle inconsistent column counts
                    if len(row) < len(headers):
                        row += [''] * (len(headers) - len(row))
                    elif len(row) > len(headers):
                        # Merge extra columns into the last column
                        row = row[:len(headers)-1] + [', '.join(row[len(headers)-1:])]
                    
                    data.append(row)
                    
            except StopIteration:
                print("File is empty")
                return pd.DataFrame()
        
        df = pd.DataFrame(data, columns=headers)
        print(f"✓ Loaded {len(df)} rows with alternative method")
        return df


In [ ]:
def load_problematic_csv(file_path):
    """Load CSV with inconsistent column counts"""
    print(f"Loading problematic CSV: {file_path}")
    

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    print(f"Total lines: {len(lines)}")
    print(f"Problematic line 68: {repr(lines[67])}") # Line 68 is index 67
  
  
    print("\nColumn counts in first 10 lines:")
    for i in range(min(10, len(lines))):
        field_count = len(lines[i].strip().split(','))
        print(f"Line {i+1}: {field_count} fields")
    
    # Now load with flexible handling
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        
        # Read header
        try:
            header = next(reader)
            print(f"\nOriginal header: {header}")
            print(f"Expected {len(header)} columns")
        except StopIteration:
            print("File is empty or cannot read header")
            return pd.DataFrame()
        
        # Process rows
        problem_rows = []
        for i, row in enumerate(reader):
            if len(row) != len(header):
                problem_rows.append((i+2, len(row), row))  # +2 for header line
                
                
                if len(row) > len(header):
                    
                    fixed_row = row[:len(header)-1] + [', '.join(row[len(header)-1:])]
                else:
                    
                    fixed_row = row + [''] * (len(header) - len(row))
                
                data.append(fixed_row)
            else:
                data.append(row)
    
  
  
    if problem_rows:
        print(f"\nFound {len(problem_rows)} problematic rows:")
        for line_num, actual_cols, row_data in problem_rows[:5]:  # Show first 5
            print(f"  Line {line_num}: Expected {len(header)} cols, got {actual_cols} cols")
            print(f"    Data: {row_data}")
    
 
 
    df = pd.DataFrame(data, columns=header)
    print(f"\n✓ Successfully loaded {len(df)} rows")
    return df



file_path = 'question_dataset.csv'
df = load_problematic_csv(file_path)

if not df.empty:
    print(f"\nDataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst 3 rows:")
    for i in range(min(3, len(df))):
        print(f"Row {i+1}:")
        for col in df.columns:
            print(f"  {col}: {df.iloc[i][col]}")
        print()

Loading problematic CSV: question_dataset.csv
Total lines: 291
Problematic line 68: 'Accounting In Business,F1, Which of the five fundamentals principles of the ACCA code of ethics and conduct imposes an obligation on members to comply with relevant laws and regulations and avoid any action that may bring discredit to the profession? This includes actions which a reasonable and informed third party, having knowledge of all relevant information, would conclude negatively affects the good reputation of the profession. A. Independence B. Objectivity C. Professional behaviour D. Professional competence and due care, Structured, multiple-choice question\n'

Column counts in first 10 lines:
Line 1: 1 fields
Line 2: 5 fields
Line 3: 1 fields
Line 4: 5 fields
Line 5: 5 fields
Line 6: 5 fields
Line 7: 5 fields
Line 8: 5 fields
Line 9: 6 fields
Line 10: 5 fields

Original header: ['High School']
Expected 1 columns

Found 149 problematic rows:
  Line 2: Expected 1 cols, got 5 cols
    Data: ['Sub

In [ ]:
try:
    # Option 1: Use on_bad_lines='skip' to skip problematic lines
    df = pd.read_csv('question_dataset.csv', on_bad_lines='skip')
    print("✓ Loaded with 'skip' option")
except Exception as e:
    print(f"Skip option failed: {e}")

try:
    # Option 2: Use Python engine 
    df = pd.read_csv('question_dataset.csv', engine='python')
    print("✓ Loaded with Python engine")
except Exception as e:
    print(f"Python engine failed: {e}")

try:
    # Option 3: Specify delimiter and quote handling
    df = pd.read_csv('question_dataset.csv', quoting=csv.QUOTE_ALL, escapechar='\\')
    print("✓ Loaded with quote handling")
except Exception as e:
    print(f"Quote handling failed: {e}")

✓ Loaded with 'skip' option
Python engine failed: Expected 5 fields in line 88, saw 7
Quote handling failed: Error tokenizing data. C error: Expected 5 fields in line 68, saw 7



In [ ]:
def parse_multi_section_csv(file_path):
    """Parse CSV file with multiple sections and different headers"""
    
    sections = {
        'high_school': [],
        'university': [],
        'other': []
    }
    
    current_section = 'other'
    current_headers = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        
        for i, row in enumerate(reader):
            if not row or all(cell.strip() == '' for cell in row):
                continue  # Skip empty rows
                
            # Check for section headers
            first_cell = row[0].strip() if row else ''
            
            if first_cell == 'High School':
                current_section = 'high_school'
                print("✓ Found High School section")
                continue
            elif first_cell == 'University':
                current_section = 'university'
                print("✓ Found University section")
                continue
            elif first_cell in ['Subject', 'Subject,Year,Question,Type,Style']:
               
                # This row contains headers
                if len(row) > 1 and ',' in row[0]:
                    

                    current_headers = [h.strip() for h in row[0].split(',')]
                else:
                   
                   
                    current_headers = [cell.strip() for cell in row if cell.strip()]
                print(f"✓ Found headers for {current_section}: {current_headers}")
                continue
            elif first_cell in ['Category', 'Category,Subject,Question,Type,Style']:
                current_section = 'other'
                if ',' in first_cell:
                    current_headers = [h.strip() for h in first_cell.split(',')]
                else:
                    current_headers = [cell.strip() for cell in row if cell.strip()]
                print(f"✓ Found headers for category section: {current_headers}")
                continue
            
            # Process data rows
            if current_headers and len(row) > 0:
                # Clean the row data
                clean_row = []
                for cell in row:
                    if cell.strip(): 
                        clean_row.append(cell.strip())
                
                if len(clean_row) >= len(current_headers):
                    # Take only the number of columns we need
                    row_data = clean_row[:len(current_headers)]
                else:
                    # Pad with empty strings if needed
                    row_data = clean_row + [''] * (len(current_headers) - len(clean_row))
                
                # Create dictionary for this row
                row_dict = dict(zip(current_headers, row_data))
                sections[current_section].append(row_dict)
    
    # Create DataFrames for each section
    dfs = {}
    for section_name, data in sections.items():
        if data:
            dfs[section_name] = pd.DataFrame(data)
            print(f"✓ {section_name}: {len(data)} rows")
        else:
            dfs[section_name] = pd.DataFrame()
            print(f"✗ {section_name}: No data")
    
    return dfs

# Parse the file
file_path = 'question_dataset.csv'
section_dfs = parse_multi_section_csv(file_path)

# Display results
for section_name, df in section_dfs.items():
    if not df.empty:
        print(f"\n=== {section_name.upper()} SECTION ===")
        print(f"Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print("\nFirst 3 rows:")
        print(df.head(3))
        print("-" * 50)

✓ Found High School section
✓ Found headers for high_school: ['Subject', 'Year', 'Question', 'Type', 'Style']
✓ Found University section
✓ Found headers for university: ['Subject', 'SubjectCode', 'Question', 'Type', 'Style']
✓ Found headers for university: ['Subject', 'SubjectCode', 'Question', 'Type', 'Style']
✓ Found headers for university: ['Subject', 'SubjectCode', 'Question', 'Type', 'Style']
✓ Found headers for category section: ['Category', 'Subject', 'Question', 'Type', 'Style']
✓ Found headers for category section: ['Category', 'Subject', 'Question', 'Type', 'Style']
✓ high_school: 60 rows
✓ university: 29 rows
✓ other: 32 rows

=== HIGH_SCHOOL SECTION ===
Shape: (60, 5)
Columns: ['Subject', 'Year', 'Question', 'Type', 'Style']

First 3 rows:
       Subject  Year                                           Question  \
0  Mathematics  2021                          Solve for x: 3x - 7 = 11.   
1  Mathematics  2021  Find the area of a triangle with base 6 cm and...   
2  Mathematic

In [ ]:
def create_clean_combined_dataset(file_path, output_path):
    """Create a clean, combined dataset from all sections"""
    
    all_data = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    current_section = None
    headers = ['Subject', 'Year', 'Question', 'Type', 'Style', 'Level']  # Standardized headers
    
    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue
            
        # Detect sections
        if line == 'High School':
            current_section = 'High School'
            continue
        elif line == 'University':
            current_section = 'University'
            continue
            
        # Skip header lines
        if 'Subject,Year,Question,Type,Style' in line or 'Subject,SubjectCode,Question,Type,Style' in line:
            continue
            
        # Process data line
        if ',' in line and current_section:
            parts = [part.strip() for part in line.split(',')]
            
            if len(parts) >= 4: 
                
                if current_section == 'High School' and len(parts) >= 5:
                    row_data = parts[:5] + [current_section]  # Subject, Year, Question, Type, Style, Level
                elif current_section == 'University' and len(parts) >= 5:
                    # University format might be different here
                    row_data = [parts[0], '', parts[2], parts[3], parts[4], current_section]
                else:
                    continue
                    
                all_data.append(row_data)
    
    # Create a clean DataFrame
    clean_df = pd.DataFrame(all_data, columns=headers)
    
    # Save theclean version
    clean_df.to_csv(output_path, index=False)
    print(f"✓ Clean combined dataset saved: {output_path}")
    print(f"✓ Total questions: {len(clean_df)}")
    print(f"✓ Levels: {clean_df['Level'].value_counts().to_dict()}")
    
    return clean_df

# Create a well needed, clean dataset
clean_df = create_clean_combined_dataset('question_dataset.csv', 'clean_question_dataset.csv')
print(clean_df.head())

✓ Clean combined dataset saved: clean_question_dataset.csv
✓ Total questions: 105
✓ Levels: {'High School': 59, 'University': 46}
       Subject  Year                                           Question  \
0  Mathematics  2021                          Solve for x: 3x - 7 = 11.   
1  Mathematics  2021  Find the area of a triangle with base 6 cm and...   
2  Mathematics  2022                     Simplify: (2x² - 8) ÷ (x - 2).   
3  Mathematics  2023  A car travels 240 km in 3 hours. Find its aver...   
4  Mathematics  2024                  Differentiate y = 3x³ - 2x² + 5x.   

         Type                       Style        Level  
0  Structured  fill-in-the-blank question  High School  
1  Structured  fill-in-the-blank question  High School  
2  Structured  fill-in-the-blank question  High School  
3  Structured  fill-in-the-blank question  High School  
4  Structured  fill-in-the-blank question  High School  


In [16]:
df = pd.read_csv('clean_question_dataset.csv')

### Step3: Here, we pre-process it for the training, aka ***clean*** ***it***

In [17]:
def preprocess_data(df):

    training_data = []

    for _, row in df.iterrows():

        context = f"Subject: {row['Subject']}, Year: {row.get('Year', 'N/A')}"
        question = row['Question']

        training_data.append({
            'context': context,
            'question': question
        })

    return training_data

### ***Step 4: Prepare dataset for model training***

In [18]:

training_data = preprocess_data(df)

### ***Step 5: Initialize model and tokenizer***

In [20]:
model_name = "t5-small" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

### ***Step 6: Tokenize data***

In [ ]:
def tokenize_function(examples):
    inputs = [f"generate question: {ctx}" for ctx in examples['context']]
    targets = examples['question']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=True)
    labels = tokenizer(targets, max_length=128, truncation=True, padding=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

#print results
print("Training data prepared!")
print(f"Total training examples: {len(training_data)}")

Training data prepared!
Total training examples: 105
